# Part-of-Speech Tagger using MLP for Universal Dependencies

In [ ]:
!pip install -q --upgrade pip
!pip uninstall -y numpy scipy
!pip install numpy==2.0.0 scipy==1.15.3
!pip install -q tensorflow datasets conllu matplotlib scikit-learn nltk gensim transformers

In [ ]:
!pip install numpy==2.0.0 scipy==1.15.3

In [ ]:
!pip install tensorflow datasets conllu matplotlib scikit-learn nltk gensim transformers

In [ ]:
# Basic libraries
import os
import io
import re
import random
import pickle
import conllu
import zipfile
import numpy as np
import pandas as pd
import urllib.request
import matplotlib.pyplot as plt

# Tensorflow imports
import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import load_model, clone_model
from tensorflow.keras.callbacks import ModelCheckpoint, Callback
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.layers import Dense, Dropout, Embedding, Flatten, Concatenate, Input

# Other imports
from collections import Counter, defaultdict
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve, accuracy_score, auc, precision_score, recall_score, f1_score

# NLTK imports
import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download("punkt_tab")

# try:
#   from gensim.models import KeyedVectors
# except ValueError:
#   !pip install --upgrade gensim tsfresh
# from gensim.models import KeyedVectors

In [ ]:
# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
# Constants
WINDOW_SIZE = 2  # Context window size (words before and after the target word)
EMBEDDING_DIM = 100  # Dimension of word embeddings
MAX_VOCAB_SIZE = 20000  # Maximum vocabulary size
BATCH_SIZE = 64
EPOCHS = 20
HIDDEN_LAYERS = [128, 64]  # Hidden layer sizes
DROPOUT_RATE = 0.3

In [ ]:
# Choose a language from Universal Dependencies
# We'll use English (EWT) as an example
LANGUAGE = "English"
TREEBANK = "EWT"

In [ ]:
# Download and load the Universal Dependencies dataset
def download_ud_dataset():
    ud_url = "https://lindat.mff.cuni.cz/repository/xmlui/bitstream/handle/11234/1-4923/ud-treebanks-v2.11.tgz"
    treebank_url = "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master"
    try:
        train_file = f"en_ewt-ud-train.conllu"
        dev_file = f"en_ewt-ud-dev.conllu"
        test_file = f"en_ewt-ud-test.conllu"
        train_url = f"{treebank_url}/{train_file}"
        dev_url = f"{treebank_url}/{dev_file}"
        test_url = f"{treebank_url}/{test_file}"
        print(f"Downloading training data from {train_url}")
        train_data = urllib.request.urlopen(train_url).read().decode('utf-8')
        print(f"Downloading development data from {dev_url}")
        dev_data = urllib.request.urlopen(dev_url).read().decode('utf-8')
        print(f"Downloading test data from {test_url}")
        test_data = urllib.request.urlopen(test_url).read().decode('utf-8')
        return train_data, dev_data, test_data
    except:
        response = urllib.request.urlopen(ud_url)
        tgz_file = response.read()
        file_stream = io.BytesIO(tgz_file)
        tar = zipfile.ZipFile(file_stream)
        treebank_folder = f"ud-treebanks-v2.11/UD_{LANGUAGE}-{TREEBANK}"
        train_file = f"{treebank_folder}/{LANGUAGE.lower()}_{TREEBANK.lower()}-ud-train.conllu"
        dev_file = f"{treebank_folder}/{LANGUAGE.lower()}_{TREEBANK.lower()}-ud-dev.conllu"
        test_file = f"{treebank_folder}/{LANGUAGE.lower()}_{TREEBANK.lower()}-ud-test.conllu"
        train_data = tar.read(train_file).decode('utf-8')
        dev_data = tar.read(dev_file).decode('utf-8')
        test_data = tar.read(test_file).decode('utf-8')
        return train_data, dev_data, test_data

In [ ]:
train_data, dev_data, test_data = download_ud_dataset()

In [ ]:
print(train_data)

In [ ]:
# Save the data to files
with open('train.conllu', 'w', encoding='utf-8') as f:
    f.write(train_data)
with open('dev.conllu', 'w', encoding='utf-8') as f:
    f.write(dev_data)
with open('test.conllu', 'w', encoding='utf-8') as f:
    f.write(test_data)

In [ ]:
# Parse CoNLL-U files
def parse_conllu(file_content):
    sentences = []
    pos_tags = []
    for sentence in conllu.parse(file_content):
        sentence_words = []
        sentence_tags = []
        for token in sentence:
            word = token['form']
            pos = token['upos']
            sentence_words.append(word)
            sentence_tags.append(pos)
        sentences.append(sentence_words)
        pos_tags.append(sentence_tags)
    return sentences, pos_tags

In [ ]:
train_sentences, train_pos_tags = parse_conllu(train_data)
dev_sentences, dev_pos_tags = parse_conllu(dev_data)
test_sentences, test_pos_tags = parse_conllu(test_data)

In [ ]:
print(train_sentences[:2])

In [ ]:
# Build vocabulary and tag set
def build_vocabulary(sentences):
    counter = Counter()
    for sentence in sentences:
        counter.update(sentence)
    vocab = ['<PAD>', '<UNK>'] + [word for word, _ in counter.most_common(MAX_VOCAB_SIZE - 2)]
    word2idx = {word: idx for idx, word in enumerate(vocab)}
    return vocab, word2idx

In [ ]:
def build_tag_set(pos_tags):
    tag_set = set()
    for sentence_tags in pos_tags:
        tag_set.update(sentence_tags)
    tag_list = sorted(list(tag_set))
    tag2idx = {tag: idx for idx, tag in enumerate(tag_list)}
    return tag_list, tag2idx

In [ ]:
vocab, word2idx = build_vocabulary(train_sentences)
tag_list, tag2idx = build_tag_set(train_pos_tags)
idx2tag = {idx: tag for tag, idx in tag2idx.items()}

In [ ]:
print(vocab[:10])
print(tag_list[:10])

In [ ]:
# Baseline model
def create_baseline_model(train_sentences, train_pos_tags, word2idx, tag2idx):
    word_tag_counts = defaultdict(Counter)
    tag_counts = Counter()
    for sentence, tags in zip(train_sentences, train_pos_tags):
        for word, tag in zip(sentence, tags):
            word_idx = word2idx.get(word, word2idx['<UNK>'])
            word_tag_counts[word_idx][tag2idx[tag]] += 1
            tag_counts[tag2idx[tag]] += 1
    word_to_tag = {word_idx: tag_counter.most_common(1)[0][0] for word_idx, tag_counter in word_tag_counts.items()}
    most_common_tag = tag_counts.most_common(1)[0][0]
    return word_to_tag, most_common_tag

In [ ]:
baseline_word_to_tag, baseline_most_common_tag = create_baseline_model(
    train_sentences, train_pos_tags, word2idx, tag2idx
)

In [ ]:
print(baseline_most_common_tag)

In [ ]:
# Embeddings
def load_pretrained_embeddings():
    return None, EMBEDDING_DIM

In [ ]:
def create_embedding_matrix(vocab, word2idx, pretrained_embeddings, embedding_dim):
    embedding_matrix = np.random.normal(scale=0.6, size=(len(vocab), embedding_dim))
    embedding_matrix[word2idx['<PAD>']] = np.zeros(embedding_dim)
    return embedding_matrix

In [ ]:
pretrained_embeddings, embedding_dim = load_pretrained_embeddings()
embedding_matrix = create_embedding_matrix(vocab, word2idx, pretrained_embeddings, embedding_dim)

In [ ]:
# Prepare data for MLP
def prepare_data_for_mlp(sentences, pos_tags, word2idx, tag2idx, window_size):
    X, y = [], []
    for sentence, tags in zip(sentences, pos_tags):
        for i, (word, tag) in enumerate(zip(sentence, tags)):
            context = []
            for j in range(i - window_size, i + window_size + 1):
                if j < 0 or j >= len(sentence) or j == i:
                    continue
                context_idx = word2idx.get(sentence[j], word2idx['<UNK>'])
                context.append(context_idx)
            while len(context) < 2 * window_size:
                context.append(word2idx['<PAD>'])
            feature = [word2idx.get(word, word2idx['<UNK>'])] + context
            X.append(feature)
            y.append(tag2idx[tag])
    return np.array(X), np.array(y)

In [ ]:
X_train, y_train = prepare_data_for_mlp(train_sentences, train_pos_tags, word2idx, tag2idx, WINDOW_SIZE)
X_dev, y_dev = prepare_data_for_mlp(dev_sentences, dev_pos_tags, word2idx, tag2idx, WINDOW_SIZE)
X_test, y_test = prepare_data_for_mlp(test_sentences, test_pos_tags, word2idx, tag2idx, WINDOW_SIZE)

In [ ]:
y_train_one_hot = to_categorical(y_train, num_classes=len(tag_list))
y_dev_one_hot = to_categorical(y_dev, num_classes=len(tag_list))
y_test_one_hot = to_categorical(y_test, num_classes=len(tag_list))

In [ ]:
print(len(y_train_one_hot))

In [ ]:
# Build and train MLP model
def build_mlp_model(vocab_size, embedding_dim, embedding_matrix, window_size, n_tags, hidden_layers, dropout_rate):
    word_input = Input(shape=(1,), dtype='int32', name='word_input')
    context_input = Input(shape=(2 * window_size,), dtype='int32', name='context_input')
    embedding_layer = Embedding(vocab_size, embedding_dim, weights=[embedding_matrix], trainable=True, name='embedding_layer')
    word_embedding = embedding_layer(word_input)
    context_embedding = embedding_layer(context_input)
    word_flat = Flatten()(word_embedding)
    context_flat = Flatten()(context_embedding)
    concat = tf.keras.layers.concatenate([word_flat, context_flat])
    x = concat
    for units in HIDDEN_LAYERS:
        x = Dense(units, activation='relu')(x)
        x = Dropout(DROPOUT_RATE)(x)
    output = Dense(n_tags, activation='softmax')(x)
    model = Model(inputs=[word_input, context_input], outputs=output)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
# Create the MLP model
mlp_model = build_mlp_model(
    vocab_size=len(vocab),
    embedding_dim=embedding_dim,
    embedding_matrix=embedding_matrix,
    window_size=WINDOW_SIZE,
    n_tags=len(tag_list),
    hidden_layers=HIDDEN_LAYERS,
    dropout_rate=DROPOUT_RATE
)

# Print model summary
mlp_model.summary()

# Reorganize input data for the model
X_train_word = X_train[:, 0:1]
X_train_context = X_train[:, 1:]
X_dev_word = X_dev[:, 0:1]
X_dev_context = X_dev[:, 1:]
X_test_word = X_test[:, 0:1]
X_test_context = X_test[:, 1:]

# Callbacks for training
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

model_checkpoint = ModelCheckpoint(
    'best_pos_tagger_model.h5',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

# Train the model
history = mlp_model.fit(
    [X_train_word, X_train_context],
    y_train_one_hot,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=([X_dev_word, X_dev_context], y_dev_one_hot),
    callbacks=[early_stopping, model_checkpoint],
    verbose=1
)

In [ ]:
# Plot loss curve
plt.figure(figsize=(10,6))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Dev Loss')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Dev Loss')
plt.grid(True)
plt.savefig('loss_curve.png')
plt.show()

In [ ]:
# Evaluate the baseline model
def evaluate_baseline(X, y, word2idx, tag2idx, word_to_tag, most_common_tag):
    """Evaluate the baseline model."""
    predictions = []
    for sample in X:
        word_idx = sample[0]
        tag_idx = word_to_tag.get(word_idx, most_common_tag)
        predictions.append(tag_idx)

    predictions = np.array(predictions)
    accuracy = np.mean(predictions == y)

    return predictions, accuracy

In [ ]:
baseline_train_preds, baseline_train_acc = evaluate_baseline(
    X_train, y_train, word2idx, tag2idx, baseline_word_to_tag, baseline_most_common_tag
)
baseline_dev_preds, baseline_dev_acc = evaluate_baseline(
    X_dev, y_dev, word2idx, tag2idx, baseline_word_to_tag, baseline_most_common_tag
)
baseline_test_preds, baseline_test_acc = evaluate_baseline(
    X_test, y_test, word2idx, tag2idx, baseline_word_to_tag, baseline_most_common_tag
)

# Evaluate the MLP model
mlp_train_preds = mlp_model.predict([X_train_word, X_train_context])
mlp_dev_preds = mlp_model.predict([X_dev_word, X_dev_context])
mlp_test_preds = mlp_model.predict([X_test_word, X_test_context])

# Get the class predictions
mlp_train_preds_classes = np.argmax(mlp_train_preds, axis=1)
mlp_dev_preds_classes = np.argmax(mlp_dev_preds, axis=1)
mlp_test_preds_classes = np.argmax(mlp_test_preds, axis=1)

In [ ]:
def calculate_metrics(y_true, y_pred, y_pred_prob=None):
    """Calculate precision, recall, F1 for each class."""
    precision = precision_score(y_true, y_pred, average=None, zero_division=0)
    recall = recall_score(y_true, y_pred, average=None, zero_division=0)
    f1 = f1_score(y_true, y_pred, average=None, zero_division=0)

    # Calculate PR AUC for each class
    pr_auc = []
    if y_pred_prob is not None:
        for i in range(y_pred_prob.shape[1]):
            precision_curve, recall_curve, _ = precision_recall_curve(
                (y_true == i).astype(int),
                y_pred_prob[:, i]
            )
            pr_auc.append(auc(recall_curve, precision_curve))
    else:
        pr_auc = [0.0] * len(precision)  # placeholder if probabilities not provided

    # Calculate macro-averages
    macro_precision = np.mean(precision)
    macro_recall = np.mean(recall)
    macro_f1 = np.mean(f1)
    macro_pr_auc = np.mean(pr_auc) if y_pred_prob is not None else 0.0

    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'pr_auc': pr_auc,
        'macro_precision': macro_precision,
        'macro_recall': macro_recall,
        'macro_f1': macro_f1,
        'macro_pr_auc': macro_pr_auc
    }


In [ ]:
# Calculate metrics for baseline model
baseline_train_metrics = calculate_metrics(y_train, baseline_train_preds)
baseline_dev_metrics = calculate_metrics(y_dev, baseline_dev_preds)
baseline_test_metrics = calculate_metrics(y_test, baseline_test_preds)

# Calculate metrics for MLP model
mlp_train_metrics = calculate_metrics(y_train, mlp_train_preds_classes, mlp_train_preds)
mlp_dev_metrics = calculate_metrics(y_dev, mlp_dev_preds_classes, mlp_dev_preds)
mlp_test_metrics = calculate_metrics(y_test, mlp_test_preds_classes, mlp_test_preds)

In [ ]:
def print_top_class_metrics(metrics, tag_list, n=10):
    """Print metrics for the top n most frequent tags."""
    # Get top n tags based on frequency
    tag_counts = Counter()
    for tag in y_train:
        tag_counts[tag] += 1

    top_tags = [tag for tag, _ in tag_counts.most_common(n)]

    print(f"\nMetrics for the {n} most frequent tags:")
    print(f"{'Tag':<10} {'Precision':<10} {'Recall':<10} {'F1':<10} {'PR-AUC':<10}")
    print("-" * 50)

    for tag_idx in top_tags:
        tag = tag_list[tag_idx]
        precision = metrics['precision'][tag_idx]
        recall = metrics['recall'][tag_idx]
        f1 = metrics['f1'][tag_idx]
        pr_auc = metrics['pr_auc'][tag_idx]

        print(f"{tag:<10} {precision:.4f}     {recall:.4f}     {f1:.4f}     {pr_auc:.4f}")

In [ ]:
def print_top_class_metrics(metrics, tag_list, n=10):
    """Print metrics for the top n most frequent tags."""
    # Get top n tags based on frequency
    tag_counts = Counter()
    for tag in y_train:
        tag_counts[tag] += 1

    top_tags = [tag for tag, _ in tag_counts.most_common(n)]

    print(f"\nMetrics for the {n} most frequent tags:")
    print(f"{'Tag':<10} {'Precision':<10} {'Recall':<10} {'F1':<10} {'PR-AUC':<10}")
    print("-" * 50)

    for tag_idx in top_tags:
        tag = tag_list[tag_idx]
        precision = metrics['precision'][tag_idx]
        recall = metrics['recall'][tag_idx]
        f1 = metrics['f1'][tag_idx]
        pr_auc = metrics['pr_auc'][tag_idx]

        print(f"{tag:<10} {precision:.4f}     {recall:.4f}     {f1:.4f}     {pr_auc:.4f}")

In [ ]:
# Print metrics for top tags
print("\nMLP Model Test Set:")
print_top_class_metrics(mlp_test_metrics, tag_list)

In [ ]:
# Visualize the confusion matrix for the most common tags
def plot_confusion_matrix(y_true, y_pred, tag_list, n=10):
    """Plot confusion matrix for the top n most frequent tags."""
    from sklearn.metrics import confusion_matrix
    import seaborn as sns
    import matplotlib.pyplot as plt

    # Get top n tags
    tag_counts = Counter()
    for tag in y_train:
        tag_counts[tag] += 1

    top_tag_indices = [tag for tag, _ in tag_counts.most_common(n)]
    top_tag_names = [tag_list[idx] for idx in top_tag_indices]

    # Filter data to only include top tags
    mask = np.isin(y_true, top_tag_indices)
    y_true_filtered = y_true[mask]
    y_pred_filtered = y_pred[mask]

    # Compute confusion matrix
    cm = confusion_matrix(y_true_filtered, y_pred_filtered, labels=top_tag_indices)

    # Plot the confusion matrix
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=top_tag_names,
                yticklabels=top_tag_names)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix for Top Tags')
    plt.tight_layout()
    plt.show()

    # Return the confusion matrix and tag information for further analysis if needed
    return cm, top_tag_indices, top_tag_names

In [ ]:
# Print metrics for top tags
print("\nBaseline Model Metrics (Test Set):")
print_top_class_metrics(baseline_test_metrics, idx2tag, n=10)

print("\nMLP Model Metrics (Test Set):")
print_top_class_metrics(mlp_test_metrics, idx2tag, n=10)

# Print overall metrics
print("\nOverall Metrics:")
print(f"{'Model':<15} {'Dataset':<10} {'Accuracy':<10} {'Macro F1':<10} {'Macro PR-AUC':<10}")
print("-" * 60)
print(f"{'Baseline':<15} {'Train':<10} {baseline_train_acc:.4f}     {baseline_train_metrics['macro_f1']:.4f}     {baseline_train_metrics['macro_pr_auc']:.4f}")
print(f"{'Baseline':<15} {'Dev':<10} {baseline_dev_acc:.4f}     {baseline_dev_metrics['macro_f1']:.4f}     {baseline_dev_metrics['macro_pr_auc']:.4f}")
print(f"{'Baseline':<15} {'Test':<10} {baseline_test_acc:.4f}     {baseline_test_metrics['macro_f1']:.4f}     {baseline_test_metrics['macro_pr_auc']:.4f}")
print(f"{'MLP':<15} {'Train':<10} {accuracy_score(y_train, mlp_train_preds_classes):.4f}     {mlp_train_metrics['macro_f1']:.4f}     {mlp_train_metrics['macro_pr_auc']:.4f}")
print(f"{'MLP':<15} {'Dev':<10} {accuracy_score(y_dev, mlp_dev_preds_classes):.4f}     {mlp_dev_metrics['macro_f1']:.4f}     {mlp_dev_metrics['macro_pr_auc']:.4f}")
print(f"{'MLP':<15} {'Test':<10} {accuracy_score(y_test, mlp_test_preds_classes):.4f}     {mlp_test_metrics['macro_f1']:.4f}     {mlp_test_metrics['macro_pr_auc']:.4f}")

# Plot confusion matrices
print("\nConfusion Matrix for Baseline Model (Test Set):")
baseline_cm, _, _ = plot_confusion_matrix(y_test, baseline_test_preds, idx2tag, n=10)

print("\nConfusion Matrix for MLP Model (Test Set):")
mlp_cm, _, _ = plot_confusion_matrix(y_test, mlp_test_preds_classes, idx2tag, n=10)

In [ ]:
# Create a directory for saving checkpoints if it doesn't exist
checkpoint_dir = 'checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

# Custom callback to save checkpoints at each epoch
class SaveEpochCheckpoint(Callback):
    def __init__(self, checkpoint_dir, model_name='model_epoch_{epoch:02d}.h5'):
        super().__init__()
        self.checkpoint_dir = checkpoint_dir
        self.model_name = model_name

    def on_epoch_end(self, epoch, logs=None):
        filepath = os.path.join(self.checkpoint_dir, self.model_name.format(epoch=epoch+1))
        self.model.save(filepath)
        print(f"\nSaved checkpoint to {filepath}")

# Function to train the model with checkpoints at each epoch
def train_model_with_checkpoints(model, X_train, y_train, X_val, y_val,
                                batch_size=64, epochs=20, checkpoint_dir='checkpoints'):
    """Train model and save checkpoints at each epoch."""
    # Clone the model to start fresh
    model_to_train = clone_model(model)
    model_to_train.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    # Set up callbacks
    checkpoint_callback = SaveEpochCheckpoint(checkpoint_dir)

    # Train the model
    history = model_to_train.fit(
        X_train, y_train,
        batch_size=batch_size,
        epochs=epochs,
        validation_data=(X_val, y_val),
        callbacks=[checkpoint_callback],
        verbose=1
    )

    # Save the final model
    final_model_path = os.path.join(checkpoint_dir, 'final_model.h5')
    model_to_train.save(final_model_path)

    # Save the training history
    with open(os.path.join(checkpoint_dir, 'training_history.pkl'), 'wb') as f:
        pickle.dump(history.history, f)

    return history, model_to_train

In [ ]:
# Function to select k best checkpoints based on validation loss
def select_best_checkpoints(history, checkpoint_dir, k=5):
    """Select the k best checkpoints based on validation loss."""
    val_losses = history.history['val_loss']

    # Get indices of k best epochs (1-indexed for filenames)
    best_epochs = np.argsort(val_losses)[:k] + 1

    # Get paths to the best checkpoint files
    best_checkpoint_paths = [
        os.path.join(checkpoint_dir, f'model_epoch_{epoch:02d}.h5')
        for epoch in best_epochs
    ]

    return best_checkpoint_paths, best_epochs

# Function to implement weight averaging ensemble
def create_weight_averaging_ensemble(model, checkpoint_paths):
    """Create an ensemble model by averaging weights from multiple checkpoints."""
    # Clone the base model architecture
    ensemble_model = clone_model(model)

    # Get the weights from each checkpoint and average them
    all_weights = []
    for path in checkpoint_paths:
        checkpoint_model = load_model(path)
        all_weights.append(checkpoint_model.get_weights())

    # Average the weights
    avg_weights = []
    for weights_list_tuples in zip(*all_weights):
        avg_weights.append(np.mean(weights_list_tuples, axis=0))

    # Set the averaged weights to the ensemble model
    ensemble_model.set_weights(avg_weights)

    # Compile the model
    ensemble_model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return ensemble_model

# Function to implement majority voting ensemble
def predict_with_majority_voting(X, checkpoint_paths, n_classes):
    """Make predictions using majority voting from multiple models."""
    # Get predictions from each model
    all_predictions = []

    for path in checkpoint_paths:
        model = load_model(path)
        predictions = model.predict(X)
        pred_classes = np.argmax(predictions, axis=1)
        all_predictions.append(pred_classes)

    # Convert to array for easier manipulation
    all_predictions = np.array(all_predictions)

    # Use majority voting to get the final prediction
    # For each sample, count the occurrences of each class
    final_predictions = []

    for i in range(all_predictions.shape[1]):
        sample_preds = all_predictions[:, i]
        # Count occurrences of each class
        counts = np.bincount(sample_preds, minlength=n_classes)
        # Get the class with the most votes
        final_predictions.append(np.argmax(counts))

    return np.array(final_predictions)

# Function to evaluate models and ensembles
def evaluate_models(X_test, y_test,
                   single_model_path,
                   weight_averaging_model,
                   majority_voting_checkpoint_paths,
                   n_classes,
                   idx2tag):
    """Evaluate and compare the performance of different models."""
    # Convert one-hot encoded targets back to class indices
    y_test_classes = np.argmax(y_test, axis=1)

    # Evaluate single best model
    single_model = load_model(single_model_path)
    single_preds = single_model.predict(X_test)
    single_preds_classes = np.argmax(single_preds, axis=1)
    single_accuracy = accuracy_score(y_test_classes, single_preds_classes)
    single_f1_macro = f1_score(y_test_classes, single_preds_classes, average='macro')

    # Evaluate weight averaging ensemble
    weight_avg_preds = weight_averaging_model.predict(X_test)
    weight_avg_preds_classes = np.argmax(weight_avg_preds, axis=1)
    weight_avg_accuracy = accuracy_score(y_test_classes, weight_avg_preds_classes)
    weight_avg_f1_macro = f1_score(y_test_classes, weight_avg_preds_classes, average='macro')

    # Evaluate majority voting ensemble
    majority_voting_preds_classes = predict_with_majority_voting(
        X_test, majority_voting_checkpoint_paths, n_classes
    )
    majority_voting_accuracy = accuracy_score(y_test_classes, majority_voting_preds_classes)
    majority_voting_f1_macro = f1_score(y_test_classes, majority_voting_preds_classes, average='macro')

    # Print results
    print("\n=== Model Performance Comparison ===")
    print(f"Single Best Model Accuracy: {single_accuracy:.4f}")
    print(f"Single Best Model F1 Macro: {single_f1_macro:.4f}")
    print(f"Weight Averaging Ensemble Accuracy: {weight_avg_accuracy:.4f}")
    print(f"Weight Averaging Ensemble F1 Macro: {weight_avg_f1_macro:.4f}")
    print(f"Majority Voting Ensemble Accuracy: {majority_voting_accuracy:.4f}")
    print(f"Majority Voting Ensemble F1 Macro: {majority_voting_f1_macro:.4f}")

    # Find some examples where ensembles perform better
    different_predictions = np.where(
        (single_preds_classes != weight_avg_preds_classes) |
        (single_preds_classes != majority_voting_preds_classes)
    )[0]

    if len(different_predictions) > 0:
        print("\n=== Example Predictions Where Models Differ ===")
        for i in np.random.choice(different_predictions, min(5, len(different_predictions)), replace=False):
            true_tag = idx2tag[y_test_classes[i]]
            single_tag = idx2tag[single_preds_classes[i]]
            weight_avg_tag = idx2tag[weight_avg_preds_classes[i]]
            majority_tag = idx2tag[majority_voting_preds_classes[i]]

            print(f"Example {i}:")
            print(f"  True tag: {true_tag}")
            print(f"  Single model predicted: {single_tag}")
            print(f"  Weight averaging predicted: {weight_avg_tag}")
            print(f"  Majority voting predicted: {majority_tag}")

    # Create a bar chart comparing the methods
    models = ['Single Model', 'Weight Averaging', 'Majority Voting']
    accuracy_scores = [single_accuracy, weight_avg_accuracy, majority_voting_accuracy]
    f1_scores = [single_f1_macro, weight_avg_f1_macro, majority_voting_f1_macro]

    x = np.arange(len(models))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 6))
    accuracy_bars = ax.bar(x - width/2, accuracy_scores, width, label='Accuracy')
    f1_bars = ax.bar(x + width/2, f1_scores, width, label='F1 Macro')

    ax.set_title('Model Performance Comparison')
    ax.set_xticks(x)
    ax.set_xticklabels(models)
    ax.set_ylim(min(min(accuracy_scores), min(f1_scores)) * 0.95, 1.0)
    ax.legend()

    # Add value labels to the bars
    def add_labels(bars):
        for bar in bars:
            height = bar.get_height()
            ax.annotate(f'{height:.4f}',
                        xy=(bar.get_x() + bar.get_width() / 2, height),
                        xytext=(0, 3),  # 3 points vertical offset
                        textcoords="offset points",
                        ha='center', va='bottom')

    add_labels(accuracy_bars)
    add_labels(f1_bars)

    plt.tight_layout()
    plt.savefig('ensemble_comparison.png')
    plt.show()

    return {
        'single_model': {
            'accuracy': single_accuracy,
            'f1_macro': single_f1_macro,
            'predictions': single_preds_classes
        },
        'weight_averaging': {
            'accuracy': weight_avg_accuracy,
            'f1_macro': weight_avg_f1_macro,
            'predictions': weight_avg_preds_classes
        },
        'majority_voting': {
            'accuracy': majority_voting_accuracy,
            'f1_macro': majority_voting_f1_macro,
            'predictions': majority_voting_preds_classes
        }
    }

# Function to analyze ensemble performance by tag
def analyze_performance_by_tag(y_true, single_preds, weight_avg_preds, majority_preds,
                              idx2tag, tag_counts, top_n=10):
    """Analyze performance of different models by tag."""
    # Get the top N tags
    top_tags = [tag for tag, _ in tag_counts.most_common(top_n)]
    top_tag_names = [idx2tag[idx] for idx in top_tags]

    # Calculate accuracy for each tag and model
    tag_accuracies = {
        'single': [],
        'weight_avg': [],
        'majority': []
    }

    for tag_idx in top_tags:
        # Get indices where the true tag is the current tag
        tag_indices = np.where(y_true == tag_idx)[0]

        if len(tag_indices) > 0:
            # Calculate accuracy for each model on this tag
            single_acc = np.mean(single_preds[tag_indices] == y_true[tag_indices])
            weight_avg_acc = np.mean(weight_avg_preds[tag_indices] == y_true[tag_indices])
            majority_acc = np.mean(majority_preds[tag_indices] == y_true[tag_indices])

            tag_accuracies['single'].append(single_acc)
            tag_accuracies['weight_avg'].append(weight_avg_acc)
            tag_accuracies['majority'].append(majority_acc)
        else:
            # No examples of this tag in the test set
            tag_accuracies['single'].append(0)
            tag_accuracies['weight_avg'].append(0)
            tag_accuracies['majority'].append(0)

    # Create a grouped bar chart
    fig, ax = plt.subplots(figsize=(14, 8))
    x = np.arange(len(top_tag_names))
    width = 0.25

    ax.bar(x - width, tag_accuracies['single'], width, label='Single Model')
    ax.bar(x, tag_accuracies['weight_avg'], width, label='Weight Averaging')
    ax.bar(x + width, tag_accuracies['majority'], width, label='Majority Voting')

    ax.set_title('Model Performance by POS Tag')
    ax.set_xlabel('POS Tag')
    ax.set_ylabel('Accuracy')
    ax.set_xticks(x)
    ax.set_xticklabels(top_tag_names, rotation=45, ha='right')
    ax.legend()
    ax.grid(True, axis='y', linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.savefig('performance_by_tag.png')
    plt.show()

# Main function to run all the ensemble methods
def run_ensemble_methods(mlp_model, X_train, y_train, X_dev, y_dev, X_test, y_test,
                        tag_list, idx2tag, k=5, epochs=10, batch_size=64):
    """Run all ensemble methods and evaluate them."""
    print("=== Training Model with Checkpoints ===")
    history, trained_model = train_model_with_checkpoints(
        mlp_model,
        [X_train_word, X_train_context], y_train_one_hot,
        [X_dev_word, X_dev_context], y_dev_one_hot,
        batch_size=batch_size,
        epochs=epochs,
        checkpoint_dir=checkpoint_dir
    )

    # Plot training history
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Model Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title('Model Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig('training_history.png')
    plt.show()

    # Select the k best checkpoints
    print(f"\n=== Selecting {k} Best Checkpoints ===")
    best_checkpoint_paths, best_epochs = select_best_checkpoints(history, checkpoint_dir, k=k)
    print(f"Best epochs: {best_epochs}")

    # Create weight averaging ensemble
    print("\n=== Creating Weight Averaging Ensemble ===")
    weight_avg_model = create_weight_averaging_ensemble(mlp_model, best_checkpoint_paths)

    # Save the weight averaging model
    weight_avg_model_path = os.path.join(checkpoint_dir, 'weight_averaging_model.h5')
    weight_avg_model.save(weight_avg_model_path)
    print(f"Weight averaging model saved to {weight_avg_model_path}")

    # Evaluate all models
    print("\n=== Evaluating Models ===")
    results = evaluate_models(
        [X_test_word, X_test_context], y_test_one_hot,
        best_checkpoint_paths[0],  # Best single model
        weight_avg_model,
        best_checkpoint_paths,     # For majority voting
        len(tag_list),
        idx2tag
    )

    # Analyze performance by tag
    y_train_labels = np.argmax(y_train, axis=1) if len(y_train.shape) > 1 else y_train
    tag_counts = Counter()
    for tag in y_train_labels:
        tag_counts[tag] += 1


    y_test_classes = np.argmax(y_test_one_hot, axis=1)
    analyze_performance_by_tag(
        y_test_classes,
        results['single_model']['predictions'],
        results['weight_averaging']['predictions'],
        results['majority_voting']['predictions'],
        idx2tag,
        tag_counts
    )

    # Return the best model based on the results
    best_model_type = max(results.keys(), key=lambda k: results[k]['accuracy'])
    print(f"\nBest model type: {best_model_type}")

    if best_model_type == 'weight_averaging':
        return weight_avg_model
    elif best_model_type == 'single_model':
        return load_model(best_checkpoint_paths[0])
    else:  # majority_voting - return the set of models
        return best_checkpoint_paths

In [ ]:
K = 5

# Run the ensemble methods
best_model = run_ensemble_methods(
    mlp_model,
    X_train_word, X_train_context,
    X_dev_word, X_dev_context,
    X_test_word, X_test_context,
    tag_list, idx2tag,
    k=K,
    epochs=10
)

In [ ]:
# Function to make predictions with the best model
def predict_with_best_model(model, X):
    """Make predictions with the best model (handles both single and ensemble models)."""
    if isinstance(model, list):
        # It's a majority voting ensemble
        return predict_with_majority_voting(X, model, len(tag_list))
    else:
        # It's a single model or weight averaging model
        predictions = model.predict(X)
        return np.argmax(predictions, axis=1)

# Make predictions on the test set with the best model
final_predictions = predict_with_best_model(
    best_model,
    [X_test_word, X_test_context]
)

# Calculate final metrics
y_test_classes = np.argmax(y_test_one_hot, axis=1)
final_accuracy = accuracy_score(y_test_classes, final_predictions)
final_f1_macro = f1_score(y_test_classes, final_predictions, average='macro')

print("\n=== Final Model Performance ===")
print(f"Accuracy: {final_accuracy:.4f}")
print(f"F1 Macro: {final_f1_macro:.4f}")


In [ ]:
# Save the final model or models
if isinstance(best_model, list):
    print("Best model is a majority voting ensemble. The models are already saved in the checkpoints directory.")
else:
    best_model.save('best_pos_tagger_ensemble.h5')
    print("Best model saved to 'best_pos_tagger_ensemble.h5'")

# Function to tag new text
def tag_text(text, model, word2idx, idx2tag, window_size):
    """Tag a new text with POS tags using the best model."""
    # Tokenize the text
    words = word_tokenize(text)

    # Prepare the input features
    X = []
    for i, word in enumerate(words):
        # Create context window
        context = []
        for j in range(i - window_size, i + window_size + 1):
            if j < 0 or j >= len(words) or j == i:
                # Skip the current word and handle out-of-bounds
                continue

            context_word = words[j]
            context_idx = word2idx.get(context_word, word2idx['<UNK>'])
            context.append(context_idx)

        # Pad the context window if needed
        while len(context) < 2 * window_size:
            context.append(word2idx['<PAD>'])

        # Add the current word
        word_idx = word2idx.get(word, word2idx['<UNK>'])

        # Form the input feature
        feature = [word_idx] + context
        X.append(feature)

    X = np.array(X)
    X_word = X[:, 0:1]
    X_context = X[:, 1:]

    # Make predictions
    if isinstance(model, list):
        # It's a majority voting ensemble
        predictions = predict_with_majority_voting([X_word, X_context], model, len(tag_list))
    else:
        # It's a single model or weight averaging model
        predictions = model.predict([X_word, X_context])
        predictions = np.argmax(predictions, axis=1)

    # Convert predictions to tags
    tags = [idx2tag[pred] for pred in predictions]

    # Return words with their tags
    return list(zip(words, tags))

In [ ]:
sample_text = "This is a text used for evaluation of the ensemble word classifier."
tagged_words = tag_text(sample_text, best_model, word2idx, idx2tag, WINDOW_SIZE)

print("\n=== Sample Text POS Tagging ===")
print(sample_text)
print(tagged_words)